# Rotation symmetry — does Neural A* plan the same problem the same way when it is turned?

A grid path-finding problem has the symmetries of the square. Turn the map, the start and
the goal together by 90°, 180° or 270° and the optimal route length does not change. A
planner with no preferred orientation returns a route of that length on every copy.

**What can break the symmetry in Neural A\*.** The search itself is orientation-free: unit
move costs and upstream's heuristic (Chebyshev + a small Euclidean tie-break) are both
invariant under quarter turns. Two things are not:

* the **encoder** — five learned 3×3 convolutions; nothing forces
  `encode(rot(map)) == rot(encode(map))`;
* **tie-breaking** among equally scored nodes, which follows row-major order. It changes
  *which* of several equally short routes comes back, not its length.

Classical A\* on the same copies is the control: whatever it does under rotation is what
the problem and the tie-breaking allow.

**What is measured, per test map and rotation** (all maps brought back to the original frame):

| quantity | meaning |
|---|---|
| `opt_ratio` | route length ÷ optimum (1.0 = shortest) |
| `d_path_cost` | route length minus the unrotated copy's |
| `route_iou` | overlap of the rotated-back route with the unrotated one (1.0 = same cells) |
| `cost_mae`, `cost_r` | the encoder's equivariance gap on free cells |

The logic lives in `src/symmetry.py` (tested in `tests/test_symmetry.py`); this notebook runs
it on the 100 test mazes and shows the results.

Run from the repository root with the project venv:
`.venv/bin/python -m jupyterlab notebooks/02_symmetry.ipynb`

In [ ]:
import sys
sys.path.insert(0, "..")

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

from scripts.demo import order_path
from src import config
from src.data import load_split
from src.exact_solver import solve_dijkstra
from src.planner import load_pretrained_neural_astar
from src.symmetry import (DEGREES, ROTATIONS, evaluate_rotations, rotate_instance,
                          rotation_study, summarize_rotations, symmetrised_cost_maps)

SEED = 0
config.set_global_seeds(SEED)
planner = load_pretrained_neural_astar()
split = load_split("test")
instances = split.instances(seed=SEED)
print(f"{len(instances)} test maps, rotations {[DEGREES[k] for k in ROTATIONS]}")

In [ ]:
WALLS = ListedColormap(["#1b2530", "#f8fafb"])


def draw(ax, design, start, goal, route=None, optimal=None, expanded=None, field=None,
         cmap="viridis", vmin=None, vmax=None, title=""):
    # Map (or a per-cell field over the free cells), routes and endpoints.
    if field is None:
        ax.imshow(design, cmap=WALLS, vmin=0, vmax=1)
    else:
        ax.imshow(np.zeros_like(design), cmap=ListedColormap(["#1b2530"]))
        ax.imshow(np.ma.masked_where(design == 0, field), cmap=cmap, vmin=vmin, vmax=vmax)
    if expanded is not None:
        ax.imshow(np.ma.masked_where(expanded == 0, expanded), cmap=ListedColormap(["#6ea0de"]), alpha=0.45)
    if optimal is not None:
        rc = np.array(optimal)
        ax.plot(rc[:, 1], rc[:, 0], color="#149e8e", lw=5, alpha=0.5)
    if route is not None:
        rc = np.array(order_path(route, start, goal))
        ax.plot(rc[:, 1], rc[:, 0], color="#f05a28", lw=2)
    ax.scatter([start[1]], [start[0]], c="#257a4d", s=50, zorder=5)
    ax.scatter([goal[1]], [goal[0]], c="#c9342a", s=50, zorder=5)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

## 1. All test maps, four rotations

One batch of four copies per map. The exact optimum is checked to be the same on every copy
(a failed check would mean the rotation itself is wrong, not the planner).

In [ ]:
rows, reports = rotation_study(planner, instances)
df = pd.DataFrame(rows)
assert (df.groupby("instance_id")["optimal_cost"].nunique() == 1).all(), "rotation changed the optimum"

TOL = 1e-6
by_rotation = df.groupby("degrees").agg(
    optimal=("opt_ratio", lambda x: (x <= 1 + TOL).mean()),
    mean_opt_ratio=("opt_ratio", "mean"),
    mean_exp_ratio=("exp_ratio", "mean"),
    same_length_as_0=("d_path_cost", lambda x: (x.abs() < TOL).mean()),
    same_route_as_0=("route_iou", lambda x: (x > 1 - 1e-9).mean()),
    classical_same_route_as_0=("classical_route_iou", lambda x: (x > 1 - 1e-9).mean()),
    cost_mae=("cost_mae", "mean"),
    cost_max=("cost_max", "mean"),
    cost_r=("cost_r", "mean"),
)
by_rotation.round(3)

In [ ]:
g = df.groupby("instance_id")
per_map = pd.DataFrame({
    "same_length": [r.same_cost for r in reports],
    "same_route": [r.same_route for r in reports],
    "classical_same_route": [r.classical_same_route for r in reports],
    "length_range": (g["path_cost"].max() - g["path_cost"].min()).values,
    "optimal_on_all": g["opt_ratio"].agg(lambda x: bool((x <= 1 + TOL).all())).values,
    "optimal_on_any": g["opt_ratio"].agg(lambda x: bool((x <= 1 + TOL).any())).values,
    "max_cost_mae": g["cost_mae"].max().values,
}, index=[r.instance_id for r in reports])

n = len(per_map)
verdict_flips = per_map["optimal_on_any"] & ~per_map["optimal_on_all"]
summary = {
    "n_maps": n,
    "maps_same_length_all_rotations": int(per_map["same_length"].sum()),
    "maps_same_route_all_rotations": int(per_map["same_route"].sum()),
    "maps_classical_same_route_all_rotations": int(per_map["classical_same_route"].sum()),
    "maps_optimal_on_all_rotations": int(per_map["optimal_on_all"].sum()),
    "maps_optimal_on_some_not_all": int(verdict_flips.sum()),
    "max_length_range": float(per_map["length_range"].max()),
    "mean_cost_mae": float(df.loc[df["k"] != 0, "cost_mae"].mean()),
    "mean_cost_r": float(df.loc[df["k"] != 0, "cost_r"].mean()),
}
for key, value in summary.items():
    print(f"{key:42} {value}")

**Reading the two tables.**

* `classical_same_route_as_0` is the control. If it is 1.0, classical A\* returns the *same*
  cells on every copy (the Euclidean tie-break settles almost every tie), so any route change
  in Neural A\* cannot be blamed on tie-breaking alone.
* `same_length_as_0 < 1` means the planner's route **length** depends on orientation — the
  strong form of asymmetry, a different answer to the same problem.
* `maps_optimal_on_some_not_all` counts maps where the project's verdict (optimal / failed)
  itself depends on how the map is turned.
* `cost_mae` and `cost_r` measure how far the encoder is from equivariant. Cost values lie in
  (0, 1); compare `cost_mae` with the typical change one cell flip causes (step 6).

In [ ]:
# Does a larger encoder gap go with a route that changes length?
gap = per_map.groupby("same_length")["max_cost_mae"].describe()[["count", "mean", "50%", "max"]]
print("largest equivariance gap (mean |Δcost| over free cells), by whether the length is stable:")
gap.round(4)

## 2. One map, up close

The map whose route length varies most across rotations (or, if none varies, the one with the
largest encoder gap). Top row: each copy as the planner sees it. Middle: the cost map rotated
back into the original frame. Bottom: its difference from the unrotated cost map.

In [ ]:
pick = int(per_map["length_range"].idxmax()) if per_map["length_range"].max() > 0 \
    else int(per_map["max_cost_mae"].idxmax())
inst = next(i for i in instances if i.instance_id == pick)
results = evaluate_rotations(planner, inst)
ref = results[0]
lim = max(np.abs(r.cost_map - ref.cost_map)[inst.map_design > 0].max() for r in results) or 1e-6

fig, axes = plt.subplots(3, 4, figsize=(15, 11.5))
for j, res in enumerate(results):
    rot = rotate_instance(inst, res.k)
    optimal = solve_dijkstra(rot.map_design, rot.start_rc, rot.goal_rc).path
    draw(axes[0, j], rot.map_design, rot.start_rc, rot.goal_rc,
         route=np.rot90(res.path_map, res.k), optimal=optimal, expanded=np.rot90(res.histories, res.k),
         title=f"{DEGREES[res.k]}°: {res.path_cost:.0f} moves (opt {res.optimal_cost:.0f}), {res.expansions:.0f} expanded")
    draw(axes[1, j], inst.map_design, inst.start_rc, inst.goal_rc, route=res.path_map,
         field=res.cost_map, cmap="Blues", vmin=0, vmax=1, title=f"cost map of the {DEGREES[res.k]}° copy, rotated back")
    draw(axes[2, j], inst.map_design, inst.start_rc, inst.goal_rc,
         field=res.cost_map - ref.cost_map, cmap="RdBu_r", vmin=-lim, vmax=lim,
         title=f"minus the 0° cost map (max |Δ| {np.abs(res.cost_map - ref.cost_map)[inst.map_design > 0].max():.3f})")
fig.suptitle(f"test map {pick}: orange = Neural A*, green = shortest route, blue = expanded", y=0.995)
fig.tight_layout()
plt.show()

## 3. Where does the encoder disagree with itself?

Per cell, the standard deviation of the four rotated-back cost maps, averaged over all test
maps (walls count as 0). A zero-padded CNN treats the border differently from the interior,
but symmetrically; a pattern that is *not* symmetric here would point at the learned filters.

In [ ]:
spread = np.mean([r.cost_spread for r in reports], axis=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
im = axes[0].imshow(spread, cmap="magma")
axes[0].set_title("mean per-cell spread over the 4 rotations (all test maps)")
plt.colorbar(im, ax=axes[0], fraction=0.046)
r = next(rep for rep in reports if rep.instance_id == pick)
im = axes[1].imshow(np.ma.masked_where(inst.map_design == 0, r.cost_spread), cmap="magma")
axes[1].set_title(f"spread on test map {pick}")
plt.colorbar(im, ax=axes[1], fraction=0.046)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
plt.show()

## 4. Control: a symmetrised encoder

Test-time augmentation over the rotation group: the encoder runs on all four rotations of the
input, the outputs are rotated back and averaged. The result is exactly equivariant by
construction (`cost_mae` ≈ 0 is the sanity check of the whole pipeline). Whatever asymmetry
remains in the routes is tie-breaking; whatever changes in optimality is what the encoder's
orientation bias was costing (or buying).

In [ ]:
rows_t, reports_t = rotation_study(planner, instances, cost_map_fn=symmetrised_cost_maps(planner))
dft = pd.DataFrame(rows_t)


def overview(frame, reps):
    return {
        "optimal (all copies)": float((frame["opt_ratio"] <= 1 + TOL).mean()),
        "mean opt ratio": float(frame["opt_ratio"].mean()),
        "mean exp ratio": float(frame["exp_ratio"].mean()),
        "maps: same length on all 4": float(np.mean([r.same_cost for r in reps])),
        "maps: same route on all 4": float(np.mean([r.same_route for r in reps])),
        "mean cost_mae (k ≠ 0)": float(frame.loc[frame["k"] != 0, "cost_mae"].mean()),
    }


comparison = pd.DataFrame({"Neural A*": overview(df, reports), "symmetrised encoder": overview(dft, reports_t)})
comparison.round(4)

## 5. Save

In [ ]:
out = {
    "summary": summary,
    "by_rotation": by_rotation.reset_index().to_dict(orient="records"),
    "symmetrised_vs_original": comparison.to_dict(),
    "illustrated_map": pick,
}
path = config.RESULTS_DIR / "symmetry.json"
path.write_text(json.dumps(out, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
df.to_csv(config.RESULTS_DIR / "symmetry_rotations.csv", index=False)
print("wrote", path)

## How to read the outcome

* **Same length everywhere, routes differ** → the planner is symmetric in what matters; the
  differences are which of several shortest routes it picks.
* **Lengths differ** → the learned cost map makes the answer depend on orientation. Check
  whether the symmetrised encoder fixes it: if it does, the asymmetry lives in the encoder,
  not in the search (the same encoder-vs-search split as step 7).
* **Verdict flips** (`maps_optimal_on_some_not_all > 0`) are the cases worth showing: the same
  problem is solved or failed depending on how it is turned. The demo's *Rotate* buttons and
  *Rotation spread* layer show them live.